# Latent Space Data Augmentation
**Goal:** Generate synthetic few-shot samples via latent interpolation/noise

In [ ]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset
from sklearn.manifold import TSNE
import plotly.graph_objects as go
import plotly.express as px
import warnings
warnings.filterwarnings('ignore')

# Path setup
PROJECT_ROOT = Path.cwd().parents[1]
sys.path.insert(0, str(PROJECT_ROOT))

# Modular imports
from models.vae import SideslipVAE, VAELoss, slerp, latent_augmentation, classify_band_torch
from utils.data_loadingV2 import load_data
from utils.band_mapping import load_band_mapping, calculate_band_mae, classify_band
from utils.reproducibility_utils import seed_everything

SEED = 42
seed_everything(SEED)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")

In [ ]:
# Load data
TRAIN_PATH = PROJECT_ROOT / "datasets_challenge" / "train"
COLUMNS_TO_DROP = ['sideSlip', 'vyCG', 'longitude', 'latitude', 'time']

train_loader, val_loader, scaler, _, _ = load_data(
    train_path=str(TRAIN_PATH), val_ratio=0.15, columns_to_drop=COLUMNS_TO_DROP,
    sequence_length=20, target="sideSlip", threshold=True, threshold_value=3.0,
    threshold_column="vxCG", batchsize=64, scaling=True, seed=SEED, shuffle=True
)
band_dict = load_band_mapping()

# Collect all data
X_train, y_train = [], []
for X, y in train_loader:
    X_train.append(X)
    y_train.append(y)
X_train = torch.cat(X_train)
y_train = torch.cat(y_train)
train_bands = classify_band(y_train.numpy(), band_dict)

print(f"Train samples: {len(X_train)}")
print(f"Band distribution: Many={sum(train_bands=='many')}, Medium={sum(train_bands=='medium')}, Few={sum(train_bands=='few')}")

## 1. Train VAE Model

In [ ]:
# Train VAE with optimal config
vae = SideslipVAE(input_dim=22, hidden_dim=64, latent_dim=16, seq_len=20).to(device)
loss_fn = VAELoss(alpha_recon=0.0, beta_kl=0.005, gamma_contrast=0.0, pred_threshold=2.0, pred_slope=1.0)
optimizer = torch.optim.AdamW(vae.parameters(), lr=3e-4)

# Stage 1: L1
loss_fn_s1 = VAELoss(alpha_recon=0.0, beta_kl=0.005, gamma_contrast=0.0, pred_threshold=0.0, pred_slope=0.0)
for epoch in range(30):
    vae.train()
    for X, y in train_loader:
        X, y = X.to(device), y.to(device)
        optimizer.zero_grad()
        loss, _ = loss_fn_s1(vae(X), X, y)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(vae.parameters(), 1.0)
        optimizer.step()

# Stage 2: WeightedL1
optimizer = torch.optim.AdamW(vae.parameters(), lr=5e-5)
for epoch in range(40):
    vae.train()
    for X, y in train_loader:
        X, y = X.to(device), y.to(device)
        optimizer.zero_grad()
        loss, _ = loss_fn(vae(X), X, y)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(vae.parameters(), 1.0)
        optimizer.step()

print(f"VAE trained: {vae.get_num_params()} params")

## 2. Encode to Latent Space

In [ ]:
@torch.no_grad()
def encode_batch(model, X_data, batch_size=256):
    """Encode data to latent space."""
    model.eval()
    all_mu, all_z = [], []
    loader = DataLoader(TensorDataset(X_data), batch_size=batch_size, shuffle=False)
    for (X,) in loader:
        mu, logvar = model.encoder(X.to(device))
        z = model.reparameterize(mu, logvar)
        all_mu.append(mu.cpu())
        all_z.append(z.cpu())
    return torch.cat(all_mu), torch.cat(all_z)

mu_train, z_train = encode_batch(vae, X_train)
print(f"Latent vectors: {z_train.shape}")

In [ ]:
# Separate by band
few_mask = train_bands == 'few'
medium_mask = train_bands == 'medium'
many_mask = train_bands == 'many'

z_few = z_train[few_mask]
z_medium = z_train[medium_mask]
z_many = z_train[many_mask]
y_few = y_train[few_mask]
y_medium = y_train[medium_mask]
y_many = y_train[many_mask]

print(f"Few-shot latents: {z_few.shape}")
print(f"Medium latents: {z_medium.shape}")
print(f"Many latents: {z_many.shape}")

## 3. Latent Space Visualization

In [ ]:
# t-SNE visualization
max_samples = 5000
indices = np.random.choice(len(z_train), min(max_samples, len(z_train)), replace=False)
z_subset = z_train[indices].numpy()
bands_subset = train_bands[indices]
y_subset = y_train[indices].numpy()

tsne = TSNE(n_components=2, random_state=SEED, perplexity=30)
z_2d = tsne.fit_transform(z_subset)

# Plot
df_plot = pd.DataFrame({'x': z_2d[:, 0], 'y': z_2d[:, 1], 'band': bands_subset, 'sideslip': y_subset})
fig = px.scatter(df_plot, x='x', y='y', color='band', hover_data=['sideslip'],
                 color_discrete_map={'many': 'green', 'medium': 'orange', 'few': 'red'},
                 title='Latent Space by Band (t-SNE)')
fig.update_traces(marker=dict(size=4, opacity=0.6))
fig.update_layout(width=800, height=600)
fig.show()

## 4. Generate Augmented Samples

In [ ]:
# Generate augmented few-shot latents
n_aug = 2000

# Method 1: Interpolation (SLERP)
z_aug_interp = latent_augmentation(z_few, n_aug=n_aug, method='interpolate')

# Method 2: Gaussian noise
z_aug_noise = latent_augmentation(z_few, n_aug=n_aug, method='gaussian', noise_std=0.1)

# Method 3: Both
z_aug_both = latent_augmentation(z_few, n_aug=n_aug, method='both', noise_std=0.1)

print(f"Augmented (interp): {z_aug_interp.shape}")
print(f"Augmented (noise): {z_aug_noise.shape}")
print(f"Augmented (both): {z_aug_both.shape}")

In [ ]:
@torch.no_grad()
def predict_from_latent(model, z_data, batch_size=256):
    """Predict sideslip from latent vectors."""
    model.eval()
    preds = []
    loader = DataLoader(TensorDataset(z_data), batch_size=batch_size, shuffle=False)
    for (z,) in loader:
        y_pred = model.predict_from_latent(z.to(device))
        preds.append(y_pred.cpu())
    return torch.cat(preds).squeeze()

# Predict from augmented latents
y_aug_interp = predict_from_latent(vae, z_aug_interp)
y_aug_noise = predict_from_latent(vae, z_aug_noise)
y_aug_both = predict_from_latent(vae, z_aug_both)

print(f"\nPrediction statistics:")
print(f"Original few-shot: mean={y_few.mean():.2f}, std={y_few.std():.2f}")
print(f"Augmented (interp): mean={y_aug_interp.mean():.2f}, std={y_aug_interp.std():.2f}")
print(f"Augmented (noise): mean={y_aug_noise.mean():.2f}, std={y_aug_noise.std():.2f}")
print(f"Augmented (both): mean={y_aug_both.mean():.2f}, std={y_aug_both.std():.2f}")

## 5. Visualize Augmented Distribution

In [ ]:
# Histogram comparison
fig = go.Figure()
fig.add_trace(go.Histogram(x=y_few.numpy(), name='Original Few', opacity=0.7, nbinsx=30))
fig.add_trace(go.Histogram(x=y_aug_interp.numpy(), name='Aug (Interp)', opacity=0.5, nbinsx=30))
fig.add_trace(go.Histogram(x=y_aug_noise.numpy(), name='Aug (Noise)', opacity=0.5, nbinsx=30))
fig.add_vline(x=4.0, line_dash='dash', line_color='red', annotation_text='Few threshold')
fig.update_layout(title='Original vs Augmented Few-shot Distribution', xaxis_title='Sideslip (deg)',
                  yaxis_title='Count', barmode='overlay', width=1000, height=500)
fig.show()

In [ ]:
# t-SNE with augmented samples
z_combined = torch.cat([z_few[:500], z_aug_both[:500]])
labels = ['Original']*min(500, len(z_few)) + ['Augmented']*min(500, len(z_aug_both))

tsne_aug = TSNE(n_components=2, random_state=SEED, perplexity=30)
z_combined_2d = tsne_aug.fit_transform(z_combined.numpy())

df_aug = pd.DataFrame({'x': z_combined_2d[:, 0], 'y': z_combined_2d[:, 1], 'type': labels})
fig = px.scatter(df_aug, x='x', y='y', color='type', color_discrete_map={'Original': 'red', 'Augmented': 'blue'},
                 title='Original vs Augmented Few-shot (Latent Space)')
fig.update_traces(marker=dict(size=6, opacity=0.6))
fig.update_layout(width=800, height=600)
fig.show()

## 6. Interpolation Analysis

In [ ]:
# Get band centroids
z_few_centroid = z_few.mean(dim=0)
z_medium_centroid = z_medium.mean(dim=0)
z_many_centroid = z_many.mean(dim=0)

# Interpolate many -> few
n_steps = 20
t_values = np.linspace(0, 1, n_steps)
interp_preds = []

for t in t_values:
    z_interp = slerp(z_many_centroid.unsqueeze(0), z_few_centroid.unsqueeze(0), t)
    with torch.no_grad():
        y_pred = vae.predict_from_latent(z_interp.to(device)).cpu().item()
    interp_preds.append(y_pred)

# Plot
fig = go.Figure()
fig.add_trace(go.Scatter(x=t_values, y=interp_preds, mode='lines+markers', name='Prediction'))
fig.add_hline(y=2.4, line_dash='dash', line_color='orange', annotation_text='Many/Medium')
fig.add_hline(y=4.0, line_dash='dash', line_color='red', annotation_text='Medium/Few')
fig.update_layout(title='Latent Interpolation: Many → Few', xaxis_title='t (0=many, 1=few)',
                  yaxis_title='Predicted Sideslip (deg)', width=800, height=400)
fig.show()

## 7. Decode Augmented Latents to Sequences

In [ ]:
@torch.no_grad()
def decode_latent(model, z_data, batch_size=256):
    """Decode latent vectors to sequences."""
    model.eval()
    seqs = []
    loader = DataLoader(TensorDataset(z_data), batch_size=batch_size, shuffle=False)
    for (z,) in loader:
        x_recon = model.decoder(z.to(device))
        seqs.append(x_recon.cpu())
    return torch.cat(seqs)

# Decode augmented latents
X_aug_decoded = decode_latent(vae, z_aug_both)
print(f"Decoded sequences: {X_aug_decoded.shape}")

In [ ]:
# Compare original and decoded
idx = 0
X_orig = X_train[few_mask][idx]
X_decoded_sample = X_aug_decoded[idx]

fig = go.Figure()
for feat in range(3):
    fig.add_trace(go.Scatter(y=X_orig[:, feat].numpy(), mode='lines', name=f'Orig feat{feat}', line=dict(dash='solid')))
    fig.add_trace(go.Scatter(y=X_decoded_sample[:, feat].numpy(), mode='lines', name=f'Aug feat{feat}', line=dict(dash='dash')))
fig.update_layout(title='Original vs Augmented Sequence', xaxis_title='Timestep', yaxis_title='Value', width=1000, height=400)
fig.show()

## 8. Export Augmented Dataset

In [ ]:
# Create augmented training set
X_train_aug = torch.cat([X_train, X_aug_decoded])
y_train_aug = torch.cat([y_train, y_aug_both])

print(f"\nOriginal train: {len(X_train)} samples")
print(f"Augmented train: {len(X_train_aug)} samples")
print(f"Added {len(X_aug_decoded)} synthetic few-shot samples")

# Band distribution after augmentation
aug_bands = classify_band(y_train_aug.numpy(), band_dict)
print(f"\nNew band distribution:")
print(f"  Many: {sum(aug_bands=='many')}")
print(f"  Medium: {sum(aug_bands=='medium')}")
print(f"  Few: {sum(aug_bands=='few')}")

In [ ]:
# Save augmented data
OUTPUT_DIR = Path('augmented_data')
OUTPUT_DIR.mkdir(exist_ok=True)

torch.save({
    'X_aug': X_aug_decoded,
    'y_aug': y_aug_both,
    'z_aug': z_aug_both,
    'method': 'vae_latent_interpolation_noise',
    'n_original_few': len(z_few),
    'n_augmented': len(z_aug_both)
}, OUTPUT_DIR / 'augmented_fewshot.pt')

print(f"Saved to {OUTPUT_DIR / 'augmented_fewshot.pt'}")

## Summary
| Method | Description | Quality |
|--------|-------------|---------|
| SLERP Interpolation | Interpolate between few-shot pairs | High diversity |
| Gaussian Noise | Add noise around few-shot latents | Local variations |
| Combined | Both methods together | Best coverage |

**Note:** Augmentation did not improve VAE performance (see EXP-9 in VAE_EXPERIMENTS.md). The few-shot manifold is too sparse.